In [4]:
import pandas as pd
import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_recall_curve, classification_report
from features import build_features

df = pd.read_csv('dataset.csv')
df = build_features(df)
df = df.dropna(subset=['дата_публикации']).sort_values('дата_публикации').reset_index(drop=True)
df.shape

(2117, 38)

In [5]:
split_idx = int(len(df) * 0.8)
train = df.iloc[:split_idx]
test = df.iloc[split_idx:]

print(f"Train: {len(train)} строк, позитивов {train['label'].sum()}")
print(f"Test:  {len(test)} строк, позитивов {test['label'].sum()}")
print(f"Период train: {train['дата_публикации'].min()} - {train['дата_публикации'].max()}")
print(f"Период test:  {test['дата_публикации'].min()} - {test['дата_публикации'].max()}")

Train: 1693 строк, позитивов 117
Test:  424 строк, позитивов 50
Период train: 2026-06-12 09:45:57 - 2026-07-07 15:11:11
Период test:  2026-07-07 15:16:00 - 2026-07-14 20:52:00


In [6]:
tfidf = TfidfVectorizer(max_features=200)
X_train_text = tfidf.fit_transform(train['название_стем'])
X_test_text = tfidf.transform(test['название_стем'])

cat_cols = ['способ_группа', 'тип_торгов']
ohe = OneHotEncoder(handle_unknown='ignore')
X_train_cat = ohe.fit_transform(train[cat_cols])
X_test_cat = ohe.transform(test[cat_cols])

num_cols = ['москва', 'нмц_указана', 'лог_цена', 'дни_до_дедлайна']
X_train_num = csr_matrix(train[num_cols].values)
X_test_num = csr_matrix(test[num_cols].values)

X_train = hstack([X_train_text, X_train_cat, X_train_num])
X_test = hstack([X_test_text, X_test_cat, X_test_num])

y_train = train['label']
y_test = test['label']

In [7]:
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, proba)
print(f"ROC AUC: {auc}")

precision, recall, thresholds = precision_recall_curve(y_test, proba)
target_recall = 0.85
idx = np.where(recall[:-1] >= target_recall)[0]
best_idx = idx[np.argmax(precision[idx])] if len(idx) else np.argmax(recall[:-1])
threshold = thresholds[best_idx]
print(f"Порог: {threshold:.3f}, precision={precision[best_idx]:.3f}, recall={recall[best_idx]:.3f}")

y_pred = (proba >= threshold).astype(int)
print(classification_report(y_test, y_pred, target_names=['не интересно', 'интересно']))

ROC AUC: 0.9009090909090909
Порог: 0.451, precision=0.405, recall=0.900
              precision    recall  f1-score   support

не интересно       0.98      0.82      0.90       374
   интересно       0.41      0.90      0.56        50

    accuracy                           0.83       424
   macro avg       0.69      0.86      0.73       424
weighted avg       0.92      0.83      0.86       424

